# Smith v. United States — Claude Opus 4.7 (3rd round, 2026-05)

Latest Anthropic flagship (`claude-opus-4-7`, generally available as of May 2026). Continues the methodology of `smith_claude.ipynb` (May 2024) and uses the same prompts, parser, and 100-completion sampling so the new data lines up with the older runs in `analyze_responses.ipynb`.

API keys are loaded from `.env` at the repo root via `python-dotenv`.

## Setup
Load libraries and the `.env` file (which lives at the repo root, one level above `code/`).

In [12]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic
from IPython.display import display, Markdown
import pandas as pd

load_dotenv(os.path.join('..', '.env'))

True

## Model + client
The family alias is pinned in `MODEL`. The probe cell below captures the exact dated snapshot the API resolves it to, so the precise micro-version used for the run is preserved in the notebook output.

In [13]:
# Latest Claude (Anthropic) GA as of 2026-05-06.
# We pin the family alias here; the exact dated snapshot the API resolves to
# is captured in the probe cell below (response.model) and preserved in cell output.
MODEL = 'claude-opus-4-7'
TEMPERATURE = 1.0           # Claude messages.create accepts temperature.
MAX_TOKENS = 2048
SYSTEM_PROMPT = 'You are an ordinary native speaker of English'

# Anthropic() picks up ANTHROPIC_API_KEY from the environment.
client = Anthropic()

In [14]:
def get_completion(prompt: str, max_tokens: int = MAX_TOKENS) -> str:
    '''Generate a Claude completion for `prompt`.'''
    response = client.messages.create(
        model=MODEL,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens,
        temperature=TEMPERATURE,
    )
    return response.content[0].text

In [15]:
# Probe call — captures the exact dated snapshot the API resolves the alias to.
probe = client.messages.create(
    model=MODEL,
    system=SYSTEM_PROMPT,
    messages=[{'role': 'user', 'content': 'Reply with one word: ok'}],
    max_tokens=10,
)
print(f'Requested model: {MODEL}')
print(f'Resolved model:  {probe.model}')
print(f'Probe content:   {probe.content[0].text!r}')

Requested model: claude-opus-4-7
Resolved model:  claude-opus-4-7
Probe content:   'ok'


## Inputs
Load the statute and the *Smith* case summary. The notebook is run from `code/`, so we reach `data/` via `os.path.join('..', 'data')`.

In [16]:
DATAPATH = os.path.join('..', 'data')

statute = open(os.path.join(DATAPATH, '18USC924c1.txt'), 'r').read()
case_summary = open(os.path.join(DATAPATH, 'smith_case_summary.txt'), 'r').read()

## Prompt with visible analysis (`prompt`) 
First variant: model is asked to *show* its reasoning inside `<legal_interpretation>` tags before giving the structured answer. Used here once for sanity / qualitative inspection — **not** used in the 100-call loop.

In [17]:
prompt = f'''
You are asked to consider a legal scenario from the perspective of an ordinary person, not a legal expert. Your task is to read a statute and a case summary, then decide whether the actions described constitute 'use' of a gun as defined by the statute.

First, please read the following statute:

<statute>
{statute}
</statute>

Now, read the following case summary:

<case_summary>
{case_summary}
</case_summary>

Before providing your final answer, please analyze the situation in detail inside <legal_interpretation> tags. Consider the facts of the case and how they relate to the language of the statute. Remember, you're approaching this as an average person, not a legal professional.

In your analysis, include the following:
1. Quote the relevant parts of the statute that define 'use' of a gun.
2. List the key actions Mr. Smith took with the gun, according to the case summary. Number each action.
3. Explain how an ordinary person might interpret these actions in relation to the statute.
4. Identify any aspects of the case that might influence your interpretation.
5. Present arguments for and against considering Mr. Smith's actions as 'use' of the gun.
6. Form your opinion on whether a reasonable person would consider that Mr. Smith 'used' his gun in the sense that the language of the statute requires.

Based on your analysis, please provide your answer in the following format:

ANSWER: [Your answer (yes or no)]
REASONING: [Your reasoning, explained in simple terms that an average person would use]

Please provide your response now.
'''

In [18]:
response = get_completion(prompt)

In [19]:
display(Markdown(response))

<legal_interpretation>
1. The statute says: "during and in relation to any crime of violence or drug trafficking crime... uses or carries a firearm." It doesn't define "use" specifically.

2. Smith's actions with the gun:
   1. Discussed his MAC-10 with a drug dealer, possibly to sell it
   2. Brought the gun in a black canvas bag to a meeting
   3. Pulled the gun out and showed it to the undercover officer
   4. Offered to trade the gun for two ounces of cocaine

3. To an ordinary person, "using a gun" typically means firing it, brandishing it as a threat, or having it ready as protection during a crime. Trading a gun like a piece of merchandise isn't what most people picture when they hear "use a firearm."

4. The gun here is the item being bartered—it's the object of the transaction, not a tool being used to accomplish something else. It's like asking whether someone "uses" a car when they sell it versus when they drive it.

5. For "use": Smith did actively employ the gun in his drug deal—without the gun, there'd be no deal. He pulled it out, displayed it, and made it central to a drug trafficking transaction. The statute says "uses or carries" broadly.

Against "use": Ordinary people understand "using a firearm" to mean using it as a weapon—firing it, threatening with it, carrying it for protection. Trading it as merchandise is a different concept. If I "use" a baseball bat, I mean I swung it, not that I sold it. Smith was using the gun as currency/commodity, not as a firearm.

6. An ordinary person hearing "uses a firearm during a drug crime" would picture someone armed with a gun while dealing drugs—using it to threaten, intimidate, or protect themselves. Not someone bartering the gun itself for drugs. The everyday meaning of "using a gun" relates to its function as a weapon.
</legal_interpretation>

ANSWER: No
REASONING: When regular people hear "used a firearm" during a drug crime, they think of someone using the gun as a weapon—pointing it at someone, firing it, or keeping it handy for protection or intimidation while doing the drug deal. Smith wasn't doing any of that. He was treating the gun like merchandise, trying to swap it for cocaine the same way someone might trade a TV or a watch. To an ordinary person, selling or trading a gun isn't really "using" it as a gun—it's just using it as something valuable to exchange. If I trade my car for drugs, most people wouldn't say I "used my car" in the drug deal in any meaningful way. The gun's special function—its ability to shoot—wasn't part of what Smith was doing here.

## Constrained prompt (`new_prompt`)
Second variant: same instructions, but the analytical steps are silent. The response is constrained to `ANSWER: <yes/no>` followed by `REASONING: <text>` so the parser can split cleanly. **This is the prompt used for the 100-call loop.**

In [20]:
new_prompt = f'''
You are asked to consider a legal scenario from the perspective of an ordinary person, not a legal expert. Your task is to read a statute and a case summary, then decide whether the actions described constitute 'use' of a gun as defined by the statute.

First, please read the following statute:

<statute>
{statute}
</statute>

Now, read the following case summary:

<case_summary>
{case_summary}
</case_summary>

Before providing your final answer, please analyze the situation in detail. Consider the facts of the case and how they relate to the language of the statute. Remember, you're approaching this as an average person, not a legal professional.

Specifically, consider the following (but DO NOT include them in your response):
1. The relevant parts of the statute that define 'use' of a gun.
2. The key actions Mr. Smith took with the gun, according to the case summary. Number each action.
3. How an ordinary person might interpret these actions in relation to the statute.
4. Any aspects of the case that might influence your interpretation.
5. Arguments for and against considering Mr. Smith's actions as 'use' of the gun.
6. Your opinion on whether a reasonable person would consider that Mr. Smith 'used' his gun in the sense that the language of the statute requires.

Based on your analysis, please provide your answer in the following format:

ANSWER: [Your answer (yes or no)]
REASONING: [Your reasoning, explained in simple terms that an average person would use]

Please provide your response now.
'''

In [21]:
test_response = get_completion(new_prompt)
display(Markdown(test_response))

Looking at this situation as an ordinary person:

The statute punishes someone who "uses or carries a firearm" during a drug trafficking crime. Smith was trying to trade his MAC-10 for cocaine - the gun was essentially his currency for buying drugs.

Smith's actions with the gun: (1) He brought the MAC-10 to the motel room in a canvas bag, (2) he discussed selling/trading it with the drug dealer, (3) he pulled it out and showed it to the undercover officer, (4) he offered to trade it for two ounces of cocaine.

When ordinary people say someone "used" a gun, they typically mean they fired it, pointed it at someone, or threatened someone with it as a weapon. That's the most natural meaning. Smith didn't do any of those things - he treated the gun like merchandise or money, not like a weapon.

On the other hand, "use" is a broad word. You can "use" a hammer to prop open a door, not just to drive nails. Smith definitely "used" the gun in some sense - he used it as a bargaining chip to get drugs. Without the gun, there would be no deal.

But I think most ordinary people, hearing that someone "used a firearm during a drug trafficking crime," would picture the gun being used as a weapon - for protection, intimidation, or violence during the drug deal. They wouldn't naturally think of someone bartering a gun for drugs as "using" it in this context. The gun here was the object being sold, not a tool being employed in its function as a firearm.

ANSWER: No
REASONING: When regular people hear "used a firearm," they think of someone using it as a weapon - shooting it, pointing it, or threatening someone with it. Smith didn't use his gun as a weapon at all. He was just trying to trade it, like you'd trade any other item, in exchange for drugs. The gun was basically being treated like money or merchandise. While you could technically say he "used" it as a bargaining chip, that's not what an ordinary person means when they say someone "used a gun" during a crime. If I told a friend "he used a gun during a drug deal," they'd assume the gun was used to threaten or shoot someone, not that it was the thing being sold.

## Quick 3-iteration parser check
Make sure the `ANSWER:` / `REASONING:` parsing handles the model's output before committing to 100 paid calls.

In [22]:
test_responses = []
test_answers = []
test_reasoning = []

for i in range(3):
    t_response = get_completion(new_prompt)
    test_answers.append(t_response[t_response.find('ANSWER:') + 7:t_response.find('REASONING:')].strip())
    test_reasoning.append(t_response[t_response.find('REASONING:') + 10:].strip())
    test_responses.append(t_response)

test_df = pd.DataFrame({'response': test_responses, 'answer': test_answers, 'reasoning': test_reasoning})
display(test_df)

,response,answer,reasoning
0,Looking at this from an ordinary person's pers...,No,"To an ordinary person, ""using a firearm"" means..."
1,Looking at this from an ordinary person's pers...,No,"When regular people hear ""used a firearm"" duri..."
2,Looking at this situation as an ordinary perso...,No,"When regular people say someone ""used a gun,"" ..."


## Generate 100 completions and save
The expensive cell. Hits the API 100 times with no rate-limit handling — expect minutes of wall time.

In [23]:
answers = []
reasoning = []

for i in range(100):
    response = get_completion(new_prompt)
    answer_start = response.find('ANSWER:') + 7
    reasoning_start = response.find('REASONING:') + 10
    answer = response[answer_start:response.find('REASONING:')].strip()
    reason = response[reasoning_start:].strip()
    answers.append(answer)
    reasoning.append(reason)

responses_df = pd.DataFrame({'answer': answers, 'reasoning': reasoning})
display(responses_df.head())

,answer,reasoning
0,No,"To an ordinary person, ""using"" a gun during a ..."
1,No,"When ordinary people say someone ""used a gun"" ..."
2,No,"When regular people talk about ""using a gun,"" ..."
3,No,"When regular people hear that someone ""used a ..."
4,No,"When ordinary people talk about ""using a gun"" ..."


## Inspect the answer distribution
If any answers come back wrapped in markdown (e.g. `Yes**\n\n**`), normalize them with a `.loc[...]` fix-up here — see the `smith_chat_gpt_2.ipynb` cell 13 pattern.

In [24]:
responses_df['answer'] = responses_df['answer'].str.strip().str.capitalize()
print(responses_df['answer'].value_counts())


answer
No     96
Yes     4
Name: count, dtype: int64


## Save
Write the cleaned DataFrame to `data/`. The CSV name encodes the exact model family for downstream `analyze_responses.ipynb` consumption.

In [25]:
OUT_CSV = os.path.join(DATAPATH, 'smith_responses_claude_opus_4_7.csv')
responses_df.to_csv(OUT_CSV, index=False)
print(f'Wrote {len(responses_df)} rows to {OUT_CSV}')

Wrote 100 rows to ../data/smith_responses_claude_opus_4_7.csv
